<a href="https://colab.research.google.com/github/ghduf0201-oss/GPT2.0-0toHero/blob/main/notebook_04_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 4 — GPT-style Dataset + Minimal Sequence Model

이제부터는 **target 구조 자체**가 바뀝니다.

기존에는 `y`가 다음 문자 1개였지만, 이제는 `y`도 sequence입니다.

- `x = [t1, t2, ..., tT]`
- `y = [t2, t3, ..., t(T+1)]`

In [1]:
# 1. 라이브러리 설치 및 임포트
!pip install -q pypdf requests

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from pypdf import PdfReader
import requests


# =====================================================================
pdf_url = "https://www.federalreserve.gov/mediacenter/files/FOMCpresconf20260617.pdf"
# =====================================================================

# 2. PDF 다운로드 및 텍스트 추출
extracted_text = ""
if pdf_url and pdf_url.startswith("http"):
    print("🚀 Downloading PDF...")
    response = requests.get(pdf_url, timeout=30)
    with open("dataset.pdf", "wb") as f:
        f.write(response.content)
    pdf_file_to_read = "dataset.pdf"
else:
    print("❌ Invalid URL.")

print("📄 Extracting text...")
reader = PdfReader(pdf_file_to_read)
for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        extracted_text += page_text + "\n"

text = extracted_text

# 3. 토크나이저 구축 및 PyTorch 텐서 변환
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)

data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

# 4. 결과 출력
print("\n=== Data Process Complete ===")
print("text length:", len(text))
print("vocab_size:", vocab_size)
print("data shape:", data.shape)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 6.5 MB/s eta 0:00:00
🚀 Downloading PDF...
📄 Extracting text...

=== Data Process Complete ===
text length: 41279
vocab_size: 77
data shape: torch.Size([41279])


## 1. GPT-style dataset

In [2]:
class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

block_size = 32
dataset = NextTokenDataset(data, block_size)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

xb, yb = next(iter(loader))
print("xb.shape:", xb.shape)
print("yb.shape:", yb.shape)

xb.shape: torch.Size([64, 32])
yb.shape: torch.Size([64, 32])


## 2. Attention 없는 최소 sequence model

In [3]:
class TinySequenceLM(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=64):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        tok = self.token_embedding(x)            # (B, T, C)
        pos = self.position_embedding(pos)[None] # (1, T, C)
        h = tok + pos
        logits = self.lm_head(h)                 # (B, T, V)
        return logits

model = TinySequenceLM(vocab_size, block_size)
logits = model(xb)
print("logits.shape:", logits.shape)

logits.shape: torch.Size([64, 32, 77])


## 3. Loss

In [4]:
def sequence_cross_entropy(logits, targets):
    return F.cross_entropy(logits.transpose(1, 2), targets)

print("initial loss:", sequence_cross_entropy(logits, yb).item())

initial loss: 4.674821853637695


## 4. 학습

In [5]:
def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = sequence_cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if max_steps is not None and step + 1 >= max_steps:
            break
    return total_loss / total_count

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TinySequenceLM(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(100):
    train_loss = train_one_epoch(model, loader, optimizer, device, max_steps=300)
    print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

epoch  0 | train loss 3.0414
epoch  1 | train loss 2.4823
epoch  2 | train loss 2.4299
epoch  3 | train loss 2.4122
epoch  4 | train loss 2.4015
epoch  5 | train loss 2.3955
epoch  6 | train loss 2.3906
epoch  7 | train loss 2.3878
epoch  8 | train loss 2.3879
epoch  9 | train loss 2.3847
epoch 10 | train loss 2.3835
epoch 11 | train loss 2.3815
epoch 12 | train loss 2.3830
epoch 13 | train loss 2.3796
epoch 14 | train loss 2.3804
epoch 15 | train loss 2.3772
epoch 16 | train loss 2.3807
epoch 17 | train loss 2.3811
epoch 18 | train loss 2.3784
epoch 19 | train loss 2.3780
epoch 20 | train loss 2.3791
epoch 21 | train loss 2.3787
epoch 22 | train loss 2.3762
epoch 23 | train loss 2.3776
epoch 24 | train loss 2.3768
epoch 25 | train loss 2.3775
epoch 26 | train loss 2.3781
epoch 27 | train loss 2.3780
epoch 28 | train loss 2.3762
epoch 29 | train loss 2.3775
epoch 30 | train loss 2.3765
epoch 31 | train loss 2.3775
epoch 32 | train loss 2.3754
epoch 33 | train loss 2.3770
epoch 34 | tra

## 5. Sampling

In [7]:
@torch.no_grad()
def sample_sequence_model(model, block_size, stoi, itos, device, start_text="Chairman Warsh:", max_new_tokens=300):
    model.eval()
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)
    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor([[stoi[ch]]], device=device)
            context = torch.cat([context[:, 1:], ix], dim=1)
    out = list(start_text)
    for _ in range(max_new_tokens):
        logits = model(context)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1)
        out.append(itos[ix.item()])
        context = torch.cat([context[:, 1:], ix], dim=1)
    return "".join(out)

print(sample_sequence_model(model, block_size, stoi, itos, device, start_text="Chairman Warsh:", max_new_tokens=400))

Chairman Warsh: tin psth t't thiord ur timar mplasss 
chande taly "cedidarth 
aldaryoverert g k LLele'd aro den whetertime CHANAI'sa hencercunkisan oy wedope boreeerd P VEnes, t frayecoley t IMARIsame torer cowh, tachert, odivertoon'sthis Thive, ngold todre, t Wern rthe bo heligiariomme war olene ome tan cy 
apr.Sille ibefon tedesinf bus ore. y, 
the dumup weefoulyould ivetrdandl,6 sechto be sis, I "ELI,  meng p


## 6. 정리

- 이제 target도 sequence입니다.
- output shape는 `(B, T, V)`입니다.
- positional embedding이 처음 도입됩니다.
- 아직 attention은 없지만 GPT의 데이터/출력 인터페이스는 이미 갖추었습니다.